## Batch Experiment Execution using the Gemini Batch API
This notebook executes the predefined prompting experiments for flaky test identification using the Gemini Batch API.The objective of this notebook is to automate the execution of all experiments while maintaining a consistent workflow for each prompting strategy.

### Experimental Workflow
Each experiment follows the same execution pipeline:

1. Upload the request JSONL file to the Gemini File API.
2. Create a Gemini Batch prediction job.
3. Monitor the batch job until completion.
4. Download the generated batch response.
5. Convert the raw Gemini response into the standardized prediction format.
6. Save the predictions for later evaluation.

In [ ]:
# ============================================================
# Imports
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types

# ============================================================
# Gemini Client
# ============================================================

from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.1-flash-lite"

# ============================================================
# Project Directories
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datasets"

REQUEST_DIR = PROJECT_ROOT / "generated_prompts" / "batch_requests"

RESULTS_DIR = PROJECT_ROOT / "results"

REQUEST_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model      : {MODEL_NAME}")
print(f"Requests   : {REQUEST_DIR}")
print(f"Results    : {RESULTS_DIR}")

Model      : gemini-3.1-flash-lite
Requests   : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\batch_requests
Results    : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results


## 2. Experiment Definitions

This study evaluates six prompting strategies to investigate the effect of prompt engineering and context augmentation on flaky test identification and classification.

Each experiment is defined by:

- Prompting strategy
- Whether contextual information is provided
- Request JSONL file
- Output directory

These experiment definitions are reused throughout the notebook to execute each experiment consistently.

In [39]:
# ============================================================
# Experiment Definitions
# ============================================================

EXPERIMENTS = [
    {
        "name": "zero_shot_without_context",
        "prompt_type": "zero_shot",
        "context_enabled": False,
    },
    {
        "name": "zero_shot_with_context",
        "prompt_type": "zero_shot",
        "context_enabled": True,
    },
    {
        "name": "zero_shot_cot_without_context",
        "prompt_type": "zero_shot_cot",
        "context_enabled": False,
    },
    {
        "name": "zero_shot_cot_with_context",
        "prompt_type": "zero_shot_cot",
        "context_enabled": True,
    },
    {
        "name": "few_shot_cot_without_context",
        "prompt_type": "few_shot_cot",
        "context_enabled": False,
    },
    {
        "name": "few_shot_cot_with_context",
        "prompt_type": "few_shot_cot",
        "context_enabled": True,
    },
]

print(f"Total Experiments: {len(EXPERIMENTS)}")

for exp in EXPERIMENTS:
    print(f"• {exp['name']}")

Total Experiments: 6
• zero_shot_without_context
• zero_shot_with_context
• zero_shot_cot_without_context
• zero_shot_cot_with_context
• few_shot_cot_without_context
• few_shot_cot_with_context


## 3. Validate Experiment Inputs

Before executing the batch experiments, this notebook verifies that all required request files are available.

Each experiment must have a corresponding JSONL request file generated during the prompt generation stage.

If any request file is missing, the execution should stop to prevent incomplete experimental results.

In [40]:
# ============================================================
# Validate Request Files
# ============================================================

print("Validating experiment request files...\n")

for experiment in EXPERIMENTS:
    request_file = REQUEST_DIR / f"{experiment['name']}.jsonl"

    if not request_file.exists():
        raise FileNotFoundError(
            f"Missing request file:\n{request_file}"
        )

    experiment["request_file"] = request_file

    print(f"✓ {experiment['name']}")

print(f"\nAll {len(EXPERIMENTS)} request files are available.")

Validating experiment request files...

✓ zero_shot_without_context
✓ zero_shot_with_context
✓ zero_shot_cot_without_context
✓ zero_shot_cot_with_context
✓ few_shot_cot_without_context
✓ few_shot_cot_with_context

All 6 request files are available.


### 4. Batch Execution Functions

This section implements the reusable functions required to execute each batch experiment.

Each experiment follows the same execution workflow:

1. Upload the request JSONL file.
2. Create a Gemini Batch job.
3. Monitor the job until completion.
4. Download the generated response.
5. Convert the response into the standardized prediction format.
6. Save the prediction results.

These functions are reused for all six experiments to ensure a consistent execution process.

## 4.1 Upload Request File

The first step in the execution workflow is uploading the generated JSONL request file to the Gemini File API.

The uploaded file is later referenced when creating the batch prediction job.

In [41]:
# ============================================================
# Upload Request File
# ============================================================

def upload_request_file(request_file: Path):
    """
    Upload a JSONL request file to the Gemini File API.

    Args:
        request_file (Path): Path to the JSONL request file.

    Returns:
        Uploaded Gemini file object.
    """

    print(f"Uploading: {request_file.name}")

    uploaded_file = client.files.upload(
        file=str(request_file),
        config=types.UploadFileConfig(
            mime_type="application/jsonl"
        ),
    )

    print(f"✓ Upload completed")
    print(f"File Name : {uploaded_file.name}")

    return uploaded_file

In [17]:
experiment = EXPERIMENTS[0]

uploaded_file = upload_request_file(experiment["request_file"])

Uploading: zero_shot_without_context.jsonl
✓ Upload completed
File Name : files/c9a6ft9xccxx


## 4.2 Create Batch Job

After uploading the request file, a Gemini Batch job is created.

The batch job references the uploaded JSONL file and specifies the model that should process every request contained in the file.

The returned batch job object is used to monitor execution and later download the generated predictions.

In [42]:
# ============================================================
# Create Batch Job
# ============================================================

def create_batch_job(uploaded_file):
    """
    Create a Gemini Batch job.

    Args:
        uploaded_file: Uploaded Gemini file object.

    Returns:
        Batch job object.
    """

    print("Creating batch job...")

    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded_file.name,
    )

    print("✓ Batch job created")
    print(f"Batch Job : {batch_job.name}")
    print(f"State     : {batch_job.state}")

    return batch_job

In [19]:
batch_job = create_batch_job(uploaded_file)

Creating batch job...
✓ Batch job created
Batch Job : batches/95rxlty6n10en2vfrll4ukeww3ei0tcadvyf
State     : JobState.JOB_STATE_PENDING


## 4.3 Monitor Batch Job

After a batch job is created, it executes asynchronously.

This function periodically checks the job status until it reaches a terminal state.

Possible states include:

- JOB_STATE_PENDING
- JOB_STATE_RUNNING
- JOB_STATE_SUCCEEDED
- JOB_STATE_FAILED
- JOB_STATE_CANCELLED

Only completed jobs are downloaded in the next step.

In [44]:
# ============================================================
# Wait for Batch Completion
# ============================================================

def wait_for_batch_completion(batch_job, poll_interval=30):
    """
    Wait until a Gemini Batch job completes.

    Args:
        batch_job: Gemini batch job object.
        poll_interval (int): Seconds between status checks.

    Returns:
        Completed batch job object.
    """

    print("\nWaiting for batch job to complete...\n")

    while True:

        batch_job = client.batches.get(name=batch_job.name)

        print(f"{time.strftime('%H:%M:%S')} | {batch_job.state}")

        if batch_job.state == "JOB_STATE_SUCCEEDED":
            print("\n✓ Batch job completed successfully.")
            return batch_job

        if batch_job.state in [
            "JOB_STATE_FAILED",
            "JOB_STATE_CANCELLED",
        ]:
            raise RuntimeError(
                f"Batch job ended with state: {batch_job.state}"
            )

        time.sleep(poll_interval)

In [52]:
completed_batch = wait_for_batch_completion(batch_job)


Waiting for batch job to complete...

21:57:23 | JobState.JOB_STATE_SUCCEEDED

✓ Batch job completed successfully.


### 4.5 Load Evaluation Dataset

Load the evaluation dataset used to generate the Gemini Batch requests.

The dataset provides the ground truth labels and metadata required to merge with Gemini's predictions after batch inference.

A lookup table is also created to enable efficient retrieval of samples using their unique identifier.

In [45]:
# ============================================================
# Load Evaluation Dataset
# ============================================================

EVALUATION_DATASET = DATA_DIR / "evaluation_dataset.jsonl"

if not EVALUATION_DATASET.exists():
    raise FileNotFoundError(
        f"Evaluation dataset not found:\n{EVALUATION_DATASET}"
    )

df = pd.read_json(EVALUATION_DATASET, lines=True)

df_lookup = df.set_index("id")

print(f"✓ Loaded {len(df)} evaluation samples.")

✓ Loaded 2210 evaluation samples.


In [14]:
df.head()

,id,test_id,isFlaky,issue_category,repo_url,issue_commit,fixed_commit,test_code,helper_methods_json,failure_log,code_under_test_json,test_code_original,helper_methods_json_original,failure_log_original,code_under_test_original,has_helper_methods,has_code_under_test,has_failure_log,context_score
0,857,ormlitecore59309e55,True,Order Dependent,https://github.com/j256/ormlite-core,59309e51c61e8a63cb5fd24a5a7607b668a5f095,c80bde196ca152ecc8a3c4f38f77dbe5a4ea3232,@Test\n\tpublic void testSetObjectCacheThrow()...,{},org.opentest4j.AssertionFailedError: Unexpecte...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,@Test\n\tpublic void testSetObjectCacheThrow()...,{},Failed Rounds: 10/10\norg.opentest4j.Assertion...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,False,True,True,3
1,1574,Closure-144-21,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.jscomp.Result': {'<ini...,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.jscomp.Result': {'<ini...,True,True,True,5
2,1483,Closure-115-5,False,Non-Flaky,https://github.com/google/closure-compiler,2d6e1c78f41248fbbb1eec43b23e7430e2bc7885,4597738e8898f738c1f969fe90479728be81cc80,public void testInlineFunctions6() {\n\n te...,"{'test': 'public void test(String js, String e...",junit.framework.AssertionFailedError:\nExpecte...,{'com.google.javascript.jscomp.DiagnosticType'...,public void testInlineFunctions6() {\n // m...,{'test': '/** * Verifies that the compiler ...,Failed Rounds: 1/1\njunit.framework.AssertionF...,{'com.google.javascript.jscomp.DiagnosticType'...,True,True,True,5
3,431,ignite3modulesstoragerocksdb19c8a82testAbortWrite,True,Implementation Dependent,https://github.com/apache/ignite-3,19c8a824bd9d31f0d0dbd3fbbdd2a32ee360cab2,de6ee0702398f9ce3022a8e265c633857f3a3d88,.\n */\n @Test\n public void testAbo...,{'read': 'protected BinaryRow read(RowId rowId...,org.junit.jupiter.api.extension.ParameterResol...,{'org.apache.ignite.internal.hlc.HybridTimesta...,.\n */\n @Test\n public void testAbo...,{'read': '/** * Reads a row. */ ...,Failed Rounds: 84/101\norg.junit.jupiter.api.e...,{'org.apache.ignite.internal.hlc.HybridTimesta...,True,True,True,5
4,1563,Closure-144-10,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.rhino.Node': {'getDoub...,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.rhino.Node': {'getDoub...,True,True,True,5


### 4.6 Download Batch Results

Download the raw output file generated by the completed Gemini Batch job.

The downloaded file contains one JSON response per request submitted to the Batch API. These responses are stored without modification so they can be parsed in the next step.

In [46]:
# ============================================================
# Download Batch Results
# ============================================================

def get_experiment_directory(experiment):
    """
    Create the output directory for an experiment.
    """

    context_folder = (
        "with_context"
        if experiment["context_enabled"]
        else "without_context"
    )

    experiment_dir = (
        RESULTS_DIR
        / MODEL_NAME
        / experiment["prompt_type"]
        / context_folder
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return experiment_dir


def download_batch_results(batch_job, experiment):
    """
    Download the raw Gemini Batch response.

    Args:
        batch_job: Completed Gemini batch job.
        experiment (dict): Experiment configuration.

    Returns:
        Path: Downloaded raw response file.
    """

    if batch_job.dest is None:
        raise RuntimeError("Batch job has no output file.")

    experiment_dir = get_experiment_directory(experiment)

    raw_response_file = experiment_dir / "raw_response.jsonl"

    output_file = client.files.get(
        name=batch_job.dest.file_name
    )

    print("Downloading batch results...")

    content = client.files.download(
        file=output_file
    )

    with open(raw_response_file, "wb") as f:
        f.write(content)

    print("✓ Download completed")
    print(f"Saved to: {raw_response_file}")

    return raw_response_file

In [58]:
batch_job = wait_for_batch_completion(batch_job)


Waiting for batch job to complete...

22:02:08 | JobState.JOB_STATE_SUCCEEDED

✓ Batch job completed successfully.


In [ ]:
raw_response_file = download_batch_results(
    batch_job,
    experiment
)

raw_response_file

### 4.7 Inspect Raw Batch Response

Before parsing the complete batch output, inspect a single response record to verify the response structure returned by the Gemini Batch API.

This validation step helps ensure the parsing logic matches the actual response format.

In [60]:
# ============================================================
# Inspect Raw Batch Response
# ============================================================

import json
from pprint import pprint

with open(raw_response_file, "r", encoding="utf-8") as f:
    first_record = json.loads(f.readline())

pprint(first_record)

{'key': '857',
 'response': {'candidates': [{'content': {'parts': [{'text': '{\n'
                                                             '    '
                                                             '"classification": '
                                                             '"Non-Flaky",\n'
                                                             '    "category": '
                                                             '"Non-Flaky",\n'
                                                             '    "reasoning": '
                                                             '"The test is a '
                                                             'standard unit '
                                                             'test using a '
                                                             'mock object to '
                                                             'verify that a '
                                                            

### 4.8 Parse Batch Responses

Parse the raw Gemini Batch responses into a standardized intermediate format.

Each response contains the unique sample identifier (`key`) together with the model's JSON prediction. This step extracts those predictions while preserving the sample identifier, enabling them to be merged with the evaluation dataset in the next step.

The parsed output contains only the model predictions and does not yet include the ground truth labels.

In [47]:
# ============================================================
# Parse Batch Responses
# ============================================================

def parse_batch_results(raw_response_file):
    """
    Parse Gemini Batch responses into an intermediate format.

    Args:
        raw_response_file (Path): Downloaded Gemini Batch response file.

    Returns:
        list: Parsed predictions.
    """

    parsed_predictions = []
    failed_predictions = []

    with open(raw_response_file, "r", encoding="utf-8") as f:

        for line_no, line in enumerate(f, start=1):

            record = json.loads(line)

            sample_id = int(record["key"])

            response_text = (
                record["response"]
                      ["candidates"][0]
                      ["content"]
                      ["parts"][0]
                      ["text"]
            )

            try:
                prediction = json.loads(response_text)

                parsed_predictions.append({
                    "id": sample_id,
                    "prediction": prediction
                })

            except json.JSONDecodeError as e:

                print("=" * 80)
                print(f"JSON parsing failed")
                print(f"Record : {line_no}")
                print(f"Sample : {sample_id}")
                print(f"Error  : {e}")
                print("=" * 80)
                print(response_text)
                print("=" * 80)

                failed_predictions.append({
                    "line": line_no,
                    "sample_id": sample_id,
                    "error": str(e),
                    "raw_response": response_text
                })

                # Skip this record and continue
                continue

    print(f"\n✓ Parsed {len(parsed_predictions)} predictions.")

    if failed_predictions:
        print(f"⚠ Skipped {len(failed_predictions)} malformed responses.")

        with open("failed_predictions.json", "w", encoding="utf-8") as fp:
            json.dump(failed_predictions, fp, indent=2)

        print("Failed responses saved to failed_predictions.json")

    return parsed_predictions

In [79]:
parsed_predictions = parse_batch_results(
    raw_response_file
)

parsed_predictions[:2]

✓ Parsed 2210 predictions.


[{'id': 857,
  'prediction': {'classification': 'Non-Flaky',
   'category': 'Non-Flaky',
   'reasoning': 'The test is a standard unit test using a mock object to verify that a RuntimeException is thrown when the underlying DAO throws a SQLException. There is no evidence of shared state, timing dependencies, or non-idempotent behavior in the provided code.',
   'evidence': ['Test Code']}},
 {'id': 1574,
  'prediction': {'classification': 'Non-Flaky',
   'category': 'Non-Flaky',
   'reasoning': 'The provided test code is a deterministic unit test that compiles and checks specific JavaScript code against an expected output string. There is no evidence of asynchronous behavior, shared state, or timing dependencies that would cause the test to pass or fail inconsistently.',
   'evidence': ['Test Code']}}]

# 4.9 Merge Predictions with Evaluation Dataset

Merge the parsed model predictions with the evaluation dataset.

This step combines the prediction generated by Gemini with the corresponding ground truth labels and metadata using the sample identifier (`id`).

The resulting records are saved in the standardized prediction format used for evaluation across all experiments.

In [48]:
# ============================================================
# Merge Predictions with Evaluation Dataset
# ============================================================

def merge_predictions(parsed_predictions, df_lookup, experiment):
    """
    Merge parsed Gemini predictions with the evaluation dataset.
    """

    merged_predictions = []
    skipped = []

    for item in parsed_predictions:

        sample_id = item["id"]
        prediction = item["prediction"]

        # Verify sample exists
        if sample_id not in df_lookup.index:
            print(f"⚠ Sample {sample_id} not found in evaluation dataset.")
            skipped.append(sample_id)
            continue

        sample = df_lookup.loc[sample_id]

        merged_predictions.append({

            "id": sample_id,

            "test_id": sample["test_id"],

            "ground_truth_classification": (
                "Flaky"
                if sample["isFlaky"]
                else "Non-Flaky"
            ),

            "ground_truth_category": sample["issue_category"],

            "predicted_classification": prediction.get("classification"),

            "predicted_category": prediction.get("category"),

            "reasoning": prediction.get("reasoning"),

            "evidence": prediction.get("evidence", []),

            "model": MODEL_NAME,

            "prompt_type": experiment["prompt_type"],

            "context_enabled": experiment["context_enabled"],

            "latency_ms": None

        })

    print(f"✓ Merged {len(merged_predictions)} predictions.")

    if skipped:
        print(f"⚠ Skipped {len(skipped)} predictions (missing sample IDs).")

    return merged_predictions

In [81]:
merged_predictions = merge_predictions(
    parsed_predictions,
    df_lookup,
    experiment
)

merged_predictions[:2]

✓ Merged 2210 predictions.


[{'id': 857,
  'test_id': 'ormlitecore59309e55',
  'ground_truth_classification': 'Flaky',
  'ground_truth_category': 'Order Dependent',
  'predicted_classification': 'Non-Flaky',
  'predicted_category': 'Non-Flaky',
  'reasoning': 'The test is a standard unit test using a mock object to verify that a RuntimeException is thrown when the underlying DAO throws a SQLException. There is no evidence of shared state, timing dependencies, or non-idempotent behavior in the provided code.',
  'evidence': ['Test Code'],
  'model': 'gemini-3.1-flash-lite',
  'prompt_type': 'zero_shot',
  'context_enabled': False,
  'latency_ms': None},
 {'id': 1574,
  'test_id': 'Closure-144-21',
  'ground_truth_classification': 'Non-Flaky',
  'ground_truth_category': 'Non-Flaky',
  'predicted_classification': 'Non-Flaky',
  'predicted_category': 'Non-Flaky',
  'reasoning': 'The provided test code is a deterministic unit test that compiles and checks specific JavaScript code against an expected output string. The

# 4.10 Save Predictions

Save the merged prediction records as a JSON Lines (`.jsonl`) file.

The resulting file serves as the standardized input for the evaluation notebook, where prediction performance is measured using classification metrics.

In [50]:
# ============================================================
# Save Predictions
# ============================================================

def save_predictions(merged_predictions, experiment):
    """
    Save merged predictions as a JSONL file.

    Args:
        merged_predictions (list): Final prediction records.
        experiment (dict): Experiment configuration.

    Returns:
        Path: Saved prediction file.
    """

    experiment_dir = get_experiment_directory(experiment)

    prediction_file = experiment_dir / "predictions.jsonl"

    with open(prediction_file, "w", encoding="utf-8") as f:

        for prediction in merged_predictions:

            f.write(
                json.dumps(
                    prediction,
                    ensure_ascii=False
                )
            )

            f.write("\n")

    print(f"✓ Saved {len(merged_predictions)} predictions.")
    print(f"Location: {prediction_file}")

    return prediction_file

In [66]:
prediction_file = save_predictions(
    merged_predictions,
    experiment
)

prediction_file

✓ Saved 2210 predictions.
Location: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\zero_shot\without_context\predictions.jsonl


WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot/without_context/predictions.jsonl')

In [52]:
# ============================================================
# Run Experiment
# ============================================================

def run_experiment(experiment):

    print("=" * 80)
    print(f"Running: {experiment['name']}")
    print("=" * 80)

    uploaded_file = upload_request_file(
        experiment["request_file"]
    )

    batch_job = create_batch_job(
        uploaded_file
    )

    batch_job = wait_for_batch_completion(
        batch_job
    )

    raw_response_file = download_batch_results(
        batch_job,
        experiment
    )

    parsed_predictions = parse_batch_results(
        raw_response_file
    )

    merged_predictions = merge_predictions(
        parsed_predictions,
        df_lookup,
        experiment
    )

    prediction_file = save_predictions(
        merged_predictions,
        experiment
    )

    print("\n✓ Experiment completed successfully.")

    return prediction_file

In [54]:
prediction_file = run_experiment(
    EXPERIMENTS[3]
)

Running: zero_shot_cot_with_context
Uploading: zero_shot_cot_with_context.jsonl
✓ Upload completed
File Name : files/2yhtwvcc4pyy
Creating batch job...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [ ]:
prediction_file = run_experiment(
    EXPERIMENTS[4]
)

Running: few_shot_cot_without_context
Uploading: few_shot_cot_without_context.jsonl
✓ Upload completed
File Name : files/8bbkj3vkgk6m
Creating batch job...
✓ Batch job created
Batch Job : batches/guge5lmk3xornv27powp3w1xwdk3hy1490k7
State     : JobState.JOB_STATE_PENDING

Waiting for batch job to complete...

23:39:25 | JobState.JOB_STATE_RUNNING
23:39:56 | JobState.JOB_STATE_RUNNING
23:40:27 | JobState.JOB_STATE_RUNNING
23:40:58 | JobState.JOB_STATE_RUNNING
23:41:28 | JobState.JOB_STATE_RUNNING
23:41:59 | JobState.JOB_STATE_RUNNING
23:42:30 | JobState.JOB_STATE_RUNNING
23:43:01 | JobState.JOB_STATE_RUNNING
23:43:32 | JobState.JOB_STATE_RUNNING
23:44:02 | JobState.JOB_STATE_RUNNING
23:44:33 | JobState.JOB_STATE_RUNNING
23:45:04 | JobState.JOB_STATE_RUNNING
23:45:35 | JobState.JOB_STATE_RUNNING
23:46:05 | JobState.JOB_STATE_RUNNING
23:46:36 | JobState.JOB_STATE_RUNNING
23:47:07 | JobState.JOB_STATE_RUNNING
23:47:38 | JobState.JOB_STATE_RUNNING
23:48:08 | JobState.JOB_STATE_SUCCEEDED

✓ 

In [30]:
prediction_file = run_experiment(
    EXPERIMENTS[5]
)

Running: few_shot_cot_with_context
Uploading: few_shot_cot_with_context.jsonl
✓ Upload completed
File Name : files/lj9s7psus98x
Creating batch job...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [34]:
# List your current batch jobs
for batch in client.batches.list():
    print(f"ID: {batch.name}, State: {batch.state}")

ID: batches/guge5lmk3xornv27powp3w1xwdk3hy1490k7, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/yx7pwnld3jj3k97a5nt39rqah3jiz0ywr40r, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/r5227col6g5qgr9wsqxmbeksvbda1vvrv4yu, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/dbn5cok48iyjwfm419n9t0anoyb7lbw8ppid, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/uydqcuyzdwnzwuulj1jjvgp6m23qs7h2y1h3, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/95rxlty6n10en2vfrll4ukeww3ei0tcadvyf, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/d8zoo5t08gygj06xuxhtrfxyfy3qtgehg9xv, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/krtyn00980tdw48rlpwd6t8e3p7ukv1w15by, State: JobState.JOB_STATE_SUCCEEDED
ID: batches/uk0bm8p90zaj6oyjpmbjy3wl3lkwab9dvhs1, State: JobState.JOB_STATE_SUCCEEDED


## 5. Run With-Context Experiments (Chunked)

The `without_context` experiments completed successfully because their total token
volume fits under the Batch API's **enqueued tokens** limit for
`gemini-3.1-flash-lite` (10,000,000 tokens on Tier 1).

The `with_context` request files are much larger, one of them totals roughly
30 million tokens, which exceeds that limit in a single batch job and causes a
`429 RESOURCE_EXHAUSTED` error at `client.batches.create(...)`.

This section adds a **chunked runner** used only for the three `with_context`
experiments:

1. Split each large request file into smaller JSONL chunks that safely fit under
   the enqueued-token limit.
2. Run each chunk through the existing upload → create → wait → download → parse
   pipeline, **one chunk at a time** (so token usage from a finished chunk is
   released before the next chunk is submitted).
3. Combine the parsed predictions from all chunks for an experiment.
4. Reuse the existing `merge_predictions` and `save_predictions` functions
   unchanged, so the final `predictions.jsonl` file is written in the same
   format and location as the `without_context` experiments.


In [63]:
# ============================================================
# Split a Large Request File into Token-Safe Chunks
# ============================================================

def split_request_file(request_file: Path, max_tokens_per_chunk: int = 8_000_000):
    """
    Split a JSONL batch request file into smaller chunk files that stay
    under the Batch API's enqueued-token limit for the model.

    Uses a rough 4-characters-per-token estimate (no API calls needed).
    Defaults to 8,000,000 tokens per chunk, leaving headroom under the
    10,000,000 token Tier 1 cap for gemini-3.1-flash-lite.

    Args:
        request_file (Path): Path to the original JSONL request file.
        max_tokens_per_chunk (int): Approx. token budget per chunk.

    Returns:
        list[Path]: Paths to the generated chunk files.
    """

    lines = request_file.read_text(encoding="utf-8").splitlines()

    chunks = []
    current_chunk = []
    current_tokens = 0

    for line in lines:
        approx_tokens = len(line) / 4

        if current_chunk and (current_tokens + approx_tokens) > max_tokens_per_chunk:
            chunks.append(current_chunk)
            current_chunk = []
            current_tokens = 0

        current_chunk.append(line)
        current_tokens += approx_tokens

    if current_chunk:
        chunks.append(current_chunk)

    chunk_paths = []

    for idx, chunk_lines in enumerate(chunks, start=1):
        chunk_path = request_file.with_name(
            f"{request_file.stem}_chunk{idx}.jsonl"
        )
        chunk_path.write_text("\n".join(chunk_lines) + "\n", encoding="utf-8")
        chunk_paths.append(chunk_path)

    print(f"Split {request_file.name} into {len(chunk_paths)} chunk(s):")
    for p in chunk_paths:
        print(f"  • {p.name}")

    return chunk_paths


### 5.1 Download Helper for Chunked Jobs

The existing `download_batch_results` function always writes to a fixed
`raw_response.jsonl` filename inside the experiment's results folder. Running
several chunks for the same experiment would overwrite that file each time, so
this helper writes each chunk's raw response to its own numbered file instead.


In [64]:
# ============================================================
# Download Batch Results for a Chunk
# ============================================================

def download_batch_results_chunk(batch_job, experiment, chunk_index):
    """
    Download a completed batch job's results to a chunk-specific file,
    so multiple chunks for the same experiment don't overwrite each other.
    """

    experiment_dir = get_experiment_directory(experiment)

    raw_response_file = experiment_dir / f"raw_response_chunk{chunk_index}.jsonl"

    output_file = client.files.get(name=batch_job.dest.file_name)

    print(f"Downloading batch results (chunk {chunk_index})...")

    content = client.files.download(file=output_file)

    with open(raw_response_file, "wb") as f:
        f.write(content)

    print(f"✓ Download completed")
    print(f"Saved to: {raw_response_file}")

    return raw_response_file


### 5.2 Combine Chunk Raw Responses into a Single File

After all chunks for an experiment finish, this concatenates their individual
`raw_response_chunkN.jsonl` files into one `raw_response.jsonl`, matching the
single file output the `without_context` experiments already produce. The
per-chunk files are kept alongside it for debugging, but the combined file is
the one that represents the experiment going forward.


In [65]:
# ============================================================
# Combine Chunk Raw Responses
# ============================================================

def combine_chunk_raw_responses(experiment, num_chunks):
    """
    Concatenate per-chunk raw response files into a single raw_response.jsonl,
    matching the output format used by the without_context experiments.
    """

    experiment_dir = get_experiment_directory(experiment)
    combined_file = experiment_dir / "raw_response.jsonl"

    with open(combined_file, "w", encoding="utf-8") as out_f:
        for chunk_index in range(1, num_chunks + 1):
            chunk_file = experiment_dir / f"raw_response_chunk{chunk_index}.jsonl"
            with open(chunk_file, "r", encoding="utf-8") as in_f:
                out_f.write(in_f.read())

    print(f"✓ Combined {num_chunks} chunk file(s) into: {combined_file}")

    return combined_file


### 5.3 Run a With-Context Experiment in Chunks

This function mirrors `run_experiment`, but processes the request file as a
series of smaller batch jobs instead of one large job, combines their raw
responses into a single file, then merges and saves, so the final output
(`raw_response.jsonl` and `predictions.jsonl`) is identical in structure and
naming to the `without_context` experiments.


In [ ]:
# ============================================================
# Run a With-Context Experiment (Chunked)
# ============================================================

def run_experiment_with_context(experiment, max_tokens_per_chunk: int = 5_000_000):

    print("=" * 80)
    print(f"Running (chunked): {experiment['name']}")
    print("=" * 80)

    chunk_files = split_request_file(
        experiment["request_file"],
        max_tokens_per_chunk=max_tokens_per_chunk,
    )

    for chunk_index, chunk_file in enumerate(chunk_files, start=1):

        print("\n" + "-" * 80)
        print(f"Chunk {chunk_index}/{len(chunk_files)}: {chunk_file.name}")
        print("-" * 80)

        uploaded_file = upload_request_file(chunk_file)

        batch_job = create_batch_job(uploaded_file)

        # Wait for this chunk to fully complete before starting the next one,
        # so its enqueued tokens are released before the next chunk is submitted.
        batch_job = wait_for_batch_completion(batch_job)

        download_batch_results_chunk(
            batch_job, experiment, chunk_index
        )

    print(f"\n✓ All {len(chunk_files)} chunk(s) complete.")

    # Combine every chunk's raw response into a single raw_response.jsonl,
    # then parse that one file — same as the without_context pipeline.
    combined_raw_file = combine_chunk_raw_responses(
        experiment, num_chunks=len(chunk_files)
    )

    all_parsed_predictions = parse_batch_results(combined_raw_file)

    print(f"Total parsed predictions: {len(all_parsed_predictions)}")

    merged_predictions = merge_predictions(
        all_parsed_predictions,
        df_lookup,
        experiment
    )

    prediction_file = save_predictions(
        merged_predictions,
        experiment
    )

    print("\n✓ With-context experiment completed successfully.")

    return prediction_file


### 5.4 Run the Three With-Context Experiments

Only the `context_enabled = True` experiments are run here, since the
`without_context` experiments already completed successfully using the
original `run_experiment` function. Each experiment is run one at a time.


In [68]:
# ============================================================
# Run All With-Context Experiments
# ============================================================

WITH_CONTEXT_EXPERIMENTS = [
    exp for exp in EXPERIMENTS if exp["context_enabled"]
]

print(f"With-context experiments to run: {len(WITH_CONTEXT_EXPERIMENTS)}")
for exp in WITH_CONTEXT_EXPERIMENTS:
    print(f"  • {exp['name']}")


With-context experiments to run: 3
  • zero_shot_with_context
  • zero_shot_cot_with_context
  • few_shot_cot_with_context


In [60]:
prediction_file = run_experiment_with_context(WITH_CONTEXT_EXPERIMENTS[0])

Running (chunked): zero_shot_with_context
Split zero_shot_with_context.jsonl into 3 chunk(s):
  • zero_shot_with_context_chunk1.jsonl
  • zero_shot_with_context_chunk2.jsonl
  • zero_shot_with_context_chunk3.jsonl

--------------------------------------------------------------------------------
Chunk 1/3: zero_shot_with_context_chunk1.jsonl
--------------------------------------------------------------------------------
Uploading: zero_shot_with_context_chunk1.jsonl
✓ Upload completed
File Name : files/r3d1dv0m430v
Creating batch job...
✓ Batch job created
Batch Job : batches/l7y6ezce4z8eml662v8jv049vamn7p7b8jct
State     : JobState.JOB_STATE_PENDING

Waiting for batch job to complete...

08:51:59 | JobState.JOB_STATE_RUNNING
08:52:30 | JobState.JOB_STATE_RUNNING
08:53:01 | JobState.JOB_STATE_RUNNING
08:53:32 | JobState.JOB_STATE_RUNNING
08:54:03 | JobState.JOB_STATE_SUCCEEDED

✓ Batch job completed successfully.
✓ Download completed
Saved to: d:\university works\Final-Year_Firts_sem\F

In [69]:
# Buffer pause before the next experiment, so the previous experiment's
# enqueued tokens have time to fully release from Google's quota ledger.
print("Waiting 90s before starting the next experiment...")
time.sleep(90)

Waiting 90s before starting the next experiment...


In [70]:
prediction_file = run_experiment_with_context(WITH_CONTEXT_EXPERIMENTS[1])

Running (chunked): zero_shot_cot_with_context
Split zero_shot_cot_with_context.jsonl into 3 chunk(s):
  • zero_shot_cot_with_context_chunk1.jsonl
  • zero_shot_cot_with_context_chunk2.jsonl
  • zero_shot_cot_with_context_chunk3.jsonl

--------------------------------------------------------------------------------
Chunk 1/3: zero_shot_cot_with_context_chunk1.jsonl
--------------------------------------------------------------------------------
Uploading: zero_shot_cot_with_context_chunk1.jsonl
✓ Upload completed
File Name : files/tkgpbsg572xr
Creating batch job...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [62]:
prediction_file = run_experiment_with_context(WITH_CONTEXT_EXPERIMENTS[2])

Running (chunked): few_shot_cot_with_context
Split few_shot_cot_with_context.jsonl into 4 chunk(s):
  • few_shot_cot_with_context_chunk1.jsonl
  • few_shot_cot_with_context_chunk2.jsonl
  • few_shot_cot_with_context_chunk3.jsonl
  • few_shot_cot_with_context_chunk4.jsonl

--------------------------------------------------------------------------------
Chunk 1/4: few_shot_cot_with_context_chunk1.jsonl
--------------------------------------------------------------------------------
Uploading: few_shot_cot_with_context_chunk1.jsonl
✓ Upload completed
File Name : files/5gwzy1zfphze
Creating batch job...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

## 6. Recover Malformed Batch Responses

This section retries only records whose original model response was not valid JSON. It never overwrites `raw_response.jsonl` or `predictions.jsonl`. Recovery artifacts are written under each experiment's `recovery/` folder, and a validated combined file is saved as `predictions_recovered.jsonl`.

Run the preparation cell first, then run `run_all_recoveries()` when ready to submit the small retry batches.

In [100]:
# ============================================================
# Safe recovery of malformed model responses
# ============================================================

RECOVERY_EXPERIMENTS = EXPERIMENTS

def get_recovery_directory(experiment):
    recovery_dir = get_experiment_directory(experiment) / 'recovery'
    recovery_dir.mkdir(parents=True, exist_ok=True)
    return recovery_dir

def get_invalid_response_ids(raw_response_file: Path):
    """Return IDs whose Gemini response text is not a JSON object."""
    invalid_ids = []
    with open(raw_response_file, encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            record = json.loads(line)
            try:
                response_text = record['response']['candidates'][0]['content']['parts'][0]['text']
                prediction = json.loads(response_text)
                if not isinstance(prediction, dict):
                    raise ValueError('Prediction is not a JSON object')
            except (KeyError, IndexError, TypeError, ValueError, json.JSONDecodeError) as error:
                invalid_ids.append(str(record['key']))
                print(f'Invalid response | line={line_no} | id={record["key"]} | {error}')
    return invalid_ids

def create_retry_request_file(experiment, overwrite=False):
    """Copy only failed original requests into a new recovery JSONL file."""
    experiment_dir = get_experiment_directory(experiment)
    recovery_dir = get_recovery_directory(experiment)
    retry_request_file = recovery_dir / 'retry_requests.jsonl'
    manifest_file = recovery_dir / 'retry_manifest.json'

    if retry_request_file.exists() and not overwrite:
        if not manifest_file.exists():
            raise FileExistsError(
                f'{retry_request_file} exists but its manifest is missing; refusing to reuse it.'
            )
        with open(manifest_file, encoding='utf-8') as f:
            manifest = json.load(f)
        retry_ids = [str(sample_id) for sample_id in manifest['sample_ids']]
        print(f'{experiment["name"]}: reusing {len(retry_ids)} prepared retry request(s)')
        return retry_request_file, retry_ids

    invalid_ids = get_invalid_response_ids(experiment_dir / 'raw_response.jsonl')
    invalid_id_set = set(invalid_ids)

    original_requests = {}
    with open(experiment['request_file'], encoding='utf-8') as f:
        for line in f:
            request = json.loads(line)
            key = str(request['key'])
            if key in invalid_id_set:
                original_requests[key] = line.rstrip('\n')

    missing_requests = invalid_id_set - set(original_requests)
    if missing_requests:
        raise RuntimeError(f'Failed IDs not found in request file: {sorted(missing_requests)}')

    with open(retry_request_file, 'w', encoding='utf-8') as f:
        for sample_id in invalid_ids:
            f.write(original_requests[sample_id] + '\n')

    manifest = {
        'experiment': experiment['name'],
        'source_raw_response': str(experiment_dir / 'raw_response.jsonl'),
        'retry_request_file': str(retry_request_file),
        'sample_ids': invalid_ids,
    }
    with open(manifest_file, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

    print(f'{experiment["name"]}: prepared {len(invalid_ids)} retry request(s)')
    return retry_request_file, invalid_ids

def download_retry_results(batch_job, experiment, overwrite=False):
    """Download retry output without touching the original raw response file."""
    retry_raw_file = get_recovery_directory(experiment) / 'raw_response_retry.jsonl'
    if retry_raw_file.exists() and not overwrite:
        raise FileExistsError(f'{retry_raw_file} already exists; refusing to overwrite it.')
    if batch_job.dest is None:
        raise RuntimeError('Retry batch job has no output file.')

    output_file = client.files.get(name=batch_job.dest.file_name)
    content = client.files.download(file=output_file)
    with open(retry_raw_file, 'wb') as f:
        f.write(content)
    return retry_raw_file

def recover_evaluation_fields(response_text):
    """Safely recover only evaluation fields from malformed model JSON."""
    import re

    def extract_string(field_name):
        match = re.search(rf'\"{field_name}\"\s*:\s*\"([^\"]*)\"', response_text)
        return match.group(1) if match else None

    classification = extract_string('classification')
    category = extract_string('category')
    reasoning_match = re.search(
        r'\"reasoning\"\s*:\s*\"(.*?)\"\s*,\s*\"evidence\"\s*:',
        response_text,
        flags=re.DOTALL,
    )
    reasoning = reasoning_match.group(1) if reasoning_match else response_text

    allowed_categories = {
        'Implementation Dependent', 'Order Dependent', 'Non-Idempotent',
        'Time Dependent', 'Non-Flaky',
    }
    if classification not in {'Flaky', 'Non-Flaky'}:
        raise ValueError(f'Invalid recovered classification: {classification!r}')
    if category not in allowed_categories:
        raise ValueError(f'Invalid recovered category: {category!r}')
    if classification == 'Non-Flaky' and category != 'Non-Flaky':
        raise ValueError('Non-Flaky classification must use Non-Flaky category.')
    if classification == 'Flaky' and category == 'Non-Flaky':
        raise ValueError('Flaky classification cannot use Non-Flaky category.')

    return {
        'classification': classification,
        'category': category,
        'reasoning': reasoning,
        'evidence': [],
    }

def parse_retry_results(retry_raw_file, expected_ids, experiment):
    """Parse retries, retaining validated evaluation fields if only JSON formatting failed."""
    parsed = []
    failures = []
    field_recoveries = []
    expected_id_set = set(expected_ids)
    with open(retry_raw_file, encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            record = json.loads(line)
            sample_id = int(record['key'])
            if str(sample_id) not in expected_id_set:
                continue
            try:
                response_text = record['response']['candidates'][0]['content']['parts'][0]['text']
                prediction = json.loads(response_text)
                if not isinstance(prediction, dict):
                    raise ValueError('Prediction is not a JSON object')
                parsed.append({'id': sample_id, 'prediction': prediction})
            except json.JSONDecodeError as error:
                try:
                    prediction = recover_evaluation_fields(response_text)
                    parsed.append({'id': sample_id, 'prediction': prediction})
                    field_recoveries.append({
                        'line': line_no, 'sample_id': sample_id, 'json_error': str(error),
                    })
                except ValueError as recovery_error:
                    failures.append({
                        'line': line_no, 'sample_id': sample_id,
                        'error': f'{error}; field recovery failed: {recovery_error}',
                    })
            except (KeyError, IndexError, TypeError, ValueError) as error:
                failures.append({'line': line_no, 'sample_id': sample_id, 'error': str(error)})

    if field_recoveries:
        audit_file = get_recovery_directory(experiment) / 'retry_field_recoveries.json'
        with open(audit_file, 'w', encoding='utf-8') as f:
            json.dump(field_recoveries, f, indent=2)
        print(f'⚠ Recovered evaluation fields from {len(field_recoveries)} malformed JSON response(s): {audit_file}')

    if failures:
        failure_file = get_recovery_directory(experiment) / 'retry_parse_failures.json'
        with open(failure_file, 'w', encoding='utf-8') as f:
            json.dump(failures, f, indent=2)
        raise RuntimeError(f'Retry produced {len(failures)} malformed response(s); details: {failure_file}')

    parsed_ids = [str(item['id']) for item in parsed]

    if len(parsed_ids) != len(set(parsed_ids)):
        raise RuntimeError('Retry output contains duplicate sample IDs.')
    if set(parsed_ids) != expected_id_set:
        raise RuntimeError(
            f'Retry validation failed. Missing={sorted(expected_id_set - set(parsed_ids))}; '
            f'Unexpected={sorted(set(parsed_ids) - expected_id_set)}'
        )
    return parsed

def write_recovered_predictions(experiment, retry_predictions, expected_ids, overwrite=False):
    """Create a new complete prediction file; leave predictions.jsonl unchanged."""
    experiment_dir = get_experiment_directory(experiment)
    original_file = experiment_dir / 'predictions.jsonl'
    recovered_file = experiment_dir / 'predictions_recovered.jsonl'

    if recovered_file.exists() and not overwrite:
        raise FileExistsError(f'{recovered_file} already exists; refusing to overwrite it.')

    with open(original_file, encoding='utf-8') as f:
        original_predictions = [json.loads(line) for line in f]

    original_ids = {str(item['id']) for item in original_predictions}
    retry_ids = {str(item['id']) for item in retry_predictions}
    expected_id_set = set(expected_ids)

    if original_ids & retry_ids:
        raise RuntimeError('Refusing to merge: a retry ID already exists in predictions.jsonl.')
    if retry_ids != expected_id_set:
        raise RuntimeError('Refusing to merge: retry prediction IDs do not match expected IDs.')

    combined = original_predictions + retry_predictions
    combined_ids = [str(item['id']) for item in combined]
    if len(combined_ids) != len(set(combined_ids)):
        raise RuntimeError('Refusing to write duplicate IDs.')
    if len(combined) != len(df_lookup):
        raise RuntimeError(
            f'Refusing to write incomplete recovery: got {len(combined)}, expected {len(df_lookup)}.'
        )

    combined.sort(key=lambda item: int(item['id']))
    with open(recovered_file, 'w', encoding='utf-8') as f:
        for item in combined:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f'✓ Wrote validated recovered file: {recovered_file}')
    return recovered_file

def run_recovery(experiment, overwrite=False):
    """Run one small retry batch and produce predictions_recovered.jsonl safely."""
    retry_request_file, expected_ids = create_retry_request_file(experiment, overwrite=overwrite)
    if not expected_ids:
        print(f'{experiment["name"]}: no recovery needed.')
        return None

    retry_raw_file = get_recovery_directory(experiment) / 'raw_response_retry.jsonl'
    if retry_raw_file.exists():
        print(f'{experiment["name"]}: using existing retry response: {retry_raw_file}')
    else:
        # Gemini's deterministic retry returned the same malformed text.
        # Recover the validated evaluation fields from the original raw response instead.
        retry_raw_file = get_experiment_directory(experiment) / 'raw_response.jsonl'
        print(f'{experiment["name"]}: using original raw response for field recovery.')
    parsed_retry = parse_retry_results(retry_raw_file, expected_ids, experiment)
    retry_predictions = merge_predictions(parsed_retry, df_lookup, experiment)
    return write_recovered_predictions(
        experiment, retry_predictions, expected_ids, overwrite=overwrite
    )

def prepare_all_recoveries(overwrite=False):
    return {
        experiment['name']: create_retry_request_file(experiment, overwrite=overwrite)
        for experiment in RECOVERY_EXPERIMENTS
    }

def run_all_recoveries(overwrite=False):
    return {
        experiment['name']: run_recovery(experiment, overwrite=overwrite)
        for experiment in RECOVERY_EXPERIMENTS
    }

# Safe first step: creates only recovery/retry_requests.jsonl and retry_manifest.json.
retry_plan = prepare_all_recoveries()
retry_plan



zero_shot_without_context: reusing 0 prepared retry request(s)
zero_shot_with_context: reusing 3 prepared retry request(s)
zero_shot_cot_without_context: reusing 0 prepared retry request(s)
zero_shot_cot_with_context: reusing 2 prepared retry request(s)
few_shot_cot_without_context: reusing 6 prepared retry request(s)
few_shot_cot_with_context: reusing 10 prepared retry request(s)


{'zero_shot_without_context': (WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot/without_context/recovery/retry_requests.jsonl'),
  []),
 'zero_shot_with_context': (WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot/with_context/recovery/retry_requests.jsonl'),
  ['2132', '1670', '1948']),
 'zero_shot_cot_without_context': (WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot_cot/without_context/recovery/retry_requests.jsonl'),
  []),
 'zero_shot_cot_with_context': (WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot_cot/with_context/recovery/retry_requests.jsonl'),
  ['1512', '503']),
 'few_shot_cot_without_context': (WindowsPath('d:/university works/Fi

In [101]:
run_all_recoveries()

zero_shot_without_context: reusing 0 prepared retry request(s)
zero_shot_without_context: no recovery needed.
zero_shot_with_context: reusing 3 prepared retry request(s)
zero_shot_with_context: using existing retry response: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\zero_shot\with_context\recovery\raw_response_retry.jsonl
⚠ Recovered evaluation fields from 3 malformed JSON response(s): d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\zero_shot\with_context\recovery\retry_field_recoveries.json
✓ Merged 3 predictions.
✓ Wrote validated recovered file: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\zero_shot\with_context\predictions_recovered.jsonl
zero_shot_cot_without_context: reusing 0 prepared retry request(s)
zero_shot_cot_without_context: no recovery needed.
zero_shot_cot_with_context:

{'zero_shot_without_context': None,
 'zero_shot_with_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot/with_context/predictions_recovered.jsonl'),
 'zero_shot_cot_without_context': None,
 'zero_shot_cot_with_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/zero_shot_cot/with_context/predictions_recovered.jsonl'),
 'few_shot_cot_without_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/few_shot_cot/without_context/predictions_recovered.jsonl'),
 'few_shot_cot_with_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gemini-3.1-flash-lite/few_shot_cot/with_context/predictions_recovered.jsonl')}